In [ ]:
from pathlib import Path
import numpy as np
import open3d as o3d

from plant_shape_analysis.dataloaders.trackplant3D import LeafSequencesDataset
from plant_shape_analysis.data_preprocessing.leaf_meshing import mesh_leaf

from plant_shape_analysis.reconstruction import classical_methods

%load_ext autoreload
%autoreload 2

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Mesh example single leaf

In [14]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [15]:
maize_leaves = leaf_dataset.get_timeseries_by_crop('maize')
len(maize_leaves)

13

In [16]:
seq_id = 4
timepoint = 0
leaf = maize_leaves[seq_id]['timepoints'][timepoint]
leaf

{'day': 0,
 'leaf_id': 1,
 'points': array([[ 7.49285 ,  2.4196  ,  4.74595 ],
        [-2.30435 , -5.14545 , 10.68065 ],
        [ 3.08935 ,  4.51075 , 16.38125 ],
        ...,
        [ 3.11065 , -3.856466,  2.507767],
        [ 1.483083, -4.716806,  6.892761],
        [-2.908264, -3.121622, 10.488623]], shape=(5505, 3)),
 'dense_points': array([[ 2.918297,  0.579248, -4.806075],
        [ 2.923797,  0.493648, -4.836475],
        [ 2.937097,  0.405248, -4.870575],
        ...,
        [ 7.092597,  1.064048, -4.190175],
        [ 7.015697,  0.558348, -4.116075],
        [ 7.016697,  0.679648, -4.171075]], shape=(98638, 3)),
 'leaf_tip': None,
 'file_path': PosixPath('../data/TrackPlant3D/gt_corrected_v1/maize/19_maize_control_plant3_D00.txt')}

In [36]:
pdcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="pymeshlab")
mesh.get_surface_area()

455.68287248649665

In [37]:
pcd, mesh = mesh_leaf(leaf['points'], method="ball_pivoting", backend="open3d")
mesh.get_surface_area()

411.92548276395524

In [38]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="pymeshlab")
mesh.get_surface_area()

1152.6762966940144

In [39]:
pcd, mesh = mesh_leaf(leaf['points'], method="poisson", backend="open3d")
mesh.get_surface_area()

1154.1699318859846

In [21]:
o3d.visualization.draw_geometries(
[mesh],
point_show_normal=True,
window_name="Leaf Point Cloud and normals",
)

## Extract all dense leaves

In [ ]:
dataset_path = Path("../data/TrackPlant3D")
leaf_dataset = LeafSequencesDataset(
    dataset_path, 
    min_timepoints=1, # we get all leaves
)

In [ ]:
# for seq in leaf_dataset:
#     for scan in seq['timepoints']:
#         if scan['dense_points'] is not None:
#             # save leaf
#             crop = ''.join([c for c in seq['sequence_name'].split('_')[0] if c.isalpha()])
#             file_name = scan['file_path'].name.replace(".txt", f"_leaf{scan['leaf_id']}_dense.ply")
#             pcd = o3d.geometry.PointCloud()
#             pcd.points = o3d.utility.Vector3dVector(scan['dense_points'])
#             save_dir = Path(f"../data/TrackPlant3D/dense_leaves/{crop}")
#             save_dir.mkdir(parents=True, exist_ok=True)
#             o3d.io.write_point_cloud(save_dir / file_name, pcd)

## Classical methods for surface reconstruction

In [23]:
# Load and visualize a sample leaf point cloud
sample_leaf_path = Path("../data/TrackPlant3D/dense_leaves/maize/19_maize_control_plant3_D00_leaf1_dense.ply")
leaf_pcd = o3d.io.read_point_cloud(str(sample_leaf_path))
leaf_pcd.estimate_normals()
leaf_pcd.orient_normals_to_align_with_direction()
leaf_pcd.orient_normals_consistent_tangent_plane(k=50)
leaf_np = np.asarray(leaf_pcd.points)
point_cloud = np.hstack([leaf_np, np.asarray(leaf_pcd.normals)])
print(f"Loaded leaf point cloud with {leaf_np.shape[0]} points.")
o3d.visualization.draw_geometries([leaf_pcd])

Loaded leaf point cloud with 98638 points.


### Apha shapes

In [24]:
mesh_alpha = classical_methods.alpha_shape_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_alpha]) # -> Not bad but has holes
mesh_alpha.get_surface_area()

831.5639802682966

### Bezier

In [25]:
mesh_bezier = classical_methods.bezier_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bezier]) # -> Has some issues, needs tuning
mesh_bezier.get_surface_area()

274.74996486034433

### B-spline

In [26]:
mesh_bspline = classical_methods.bspline_surface_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_bspline]) # -> Looks better than bezier, but has some singularities
mesh_bspline.get_surface_area()

290.7068008360359

### Delaunay

In [27]:
mesh_delaunay = classical_methods.delaunay_mesh_reconstruction(leaf_np)
o3d.visualization.draw_geometries([mesh_delaunay])
mesh_delaunay.get_surface_area()

56966.61674267357

### Poisson

In [28]:
mesh_poisson = classical_methods.poisson_reconstruction(leaf_np, depth=9)
o3d.visualization.draw_geometries([mesh_poisson])
mesh_poisson.get_surface_area()

1187.5404845772248

### Ball pivoting

In [29]:
mesh_bpa = classical_methods.ball_pivoting_reconstruction(leaf_np, radius_multiplier=5.0)
o3d.visualization.draw_geometries([mesh_bpa])
mesh_bpa.get_surface_area()

422.9976274607068

Alternatively the area could be computed as below but it gives the same results

In [30]:
vertices_bezier = np.asarray(mesh_bezier.vertices)
triangles_bezier = np.asarray(mesh_bezier.triangles)

area_bezier_manual = classical_methods.compute_mesh_area(vertices_bezier, triangles_bezier)
float(area_bezier_manual)

274.74996486034433